In [1]:
# install dependencies 
print("Installing dependencies...")
import pandas as pd
import numpy as np 
import sqlite3
import sqlalchemy
from dotenv import load_dotenv
load_dotenv()


import os 
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import lightgbm as lgb 

from typing import TypedDict, List, Optional, Annotated
from langgraph.graph.message import add_messages
import operator

import logging
import sys
from pathlib import Path
from logging.handlers import RotatingFileHandler
import joblib
from pprint import pformat
import time

from langchain_core.prompts import PromptTemplate
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.prompt import SQL_PREFIX,SQL_SUFFIX
from langchain.agents import create_agent
from langchain.chat_models import BaseChatModel
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.utils.uuid import uuid7
from langchain_community.callbacks.manager import get_openai_callback

def get_logger(
    name: str = __name__,
    log_file: str = "app.log",
    level: int = logging.DEBUG,
    max_bytes: int = 5 * 1024 * 1024,
    backup_count: int = 3
) -> logging.Logger:
    """
    Buat logger yang siap pakai dengan output ke console + file.

    Args:
        name        : nama logger, biasanya __name__ dari modul yang memanggil
        log_file    : path file log
        level       : minimum level yang direkam (default: DEBUG)
        max_bytes   : ukuran max per file log sebelum di-rotate (default: 5MB)
        backup_count: jumlah file backup yang disimpan

    Returns:
        logging.Logger
    """

    # Make sure log file exists
    Path(log_file).parent.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger(name)

    # Avoid duplicate handler, if handler exists, use the existing handler
    if logger.handlers:
        return logger

    logger.setLevel(level)

    # logging format
    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(filename)s:%(lineno)d | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    # handler for console stdout
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)   # console: INFO ke atas saja
    console_handler.setFormatter(fmt)

    # handler for log file
    file_handler = RotatingFileHandler(
        log_file,
        maxBytes=max_bytes,
        backupCount=backup_count,
        encoding="utf-8"
    )
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(fmt)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

    return logger

# get root path of the project
PATH = Path(os.path.dirname(os.getcwd()))
DATA_DIR = PATH / "data" 
PROCESSED_DIR = DATA_DIR / "processed"
DATA_FILE = PROCESSED_DIR / "hdi_daily_ops_cleaned.csv"
MODEL_PATH = PATH / "models" / "lgbm.pkl"
FEATURE_STORES = PROCESSED_DIR / "feature_store" / "hdi_daily_ops_feature_store.csv"
logger = get_logger(log_file="logs/chatbot.log")

print("All dependencies installed successfully!")
print(f"Current project directory: {PATH}")

Installing dependencies...
All dependencies installed successfully!
Current project directory: /Users/nadif/projects/hdi-recruitment


In [2]:
def get_engine_from_csv(csv_path, table_name="hdi_daily_ops"):
    df = pd.read_csv(csv_path, parse_dates=["date"])

    engine = sqlalchemy.create_engine(
        "sqlite://",
        connect_args={"check_same_thread": False},
        poolclass=sqlalchemy.pool.StaticPool,
    )

    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False,
    )

    return engine

def get_db_util(db_engine):
    return SQLDatabase(db_engine)

def get_sql_tools(db:SQLDatabase, llm:BaseChatModel):
    return SQLDatabaseToolkit(db=db, llm=llm).get_tools()




In [3]:
# create tool for forecasting
def build_features(df: pd.DataFrame, feature_store_path:str = None, drop_na=True) -> pd.DataFrame:
    df = df.copy().sort_values("date").reset_index(drop=True)

    target = "new_enterpriser_count"

    # build lag features
    for lag in [1, 2, 3, 7, 14, 21, 28]:
        df[f"lag_{lag}"] = df[target].shift(lag)

    # build rolling statistics
    for window in [7, 14, 30, 90]:
        df[f"rolling_mean_{window}"] = df[target].rolling(window).mean()
        df[f"rolling_std_{window}"] = df[target].rolling(window).std()

    # calendar features
    df["day_of_week_num"] = df["date"].dt.dayofweek
    df["day_of_month"] = df["date"].dt.day
    df["month"] = df["date"].dt.month 
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df["day_of_week_num"] > 4).astype(int)
    df["quarter"] = df["date"].dt.quarter

    # create new feature from is_promo_period information
    df["promo_lag1"] = df["is_promo_period"].shift(1).fillna(0)
    df["promo_next1"] = df["is_promo_period"].shift(-1).fillna(0)

    # capture multi-years trend with days_since_start feature
    df["days_since_start"] = (df['date'] - df['date'].min()).dt.days

    # save the data to feature store
    if feature_store_path:
        df.dropna().to_csv(feature_store_path, index=False)

    if drop_na:
        return df.dropna()
    else:
        return df


@tool
def forecast_next_n_days(n_days:int = 7):
    """
    Forecast the number of new enterprisers for the next N days using a
    pre-trained LightGBM time series forecasting model.

    This tool performs recursive multi-step forecasting by using historical
    data from the feature store and feeding each predicted value back into
    the feature engineering pipeline to generate forecasts for subsequent
    days.

    Use this tool whenever the user asks for:
    - Future predictions of new enterpriser counts.
    - Expected new enterpriser numbers over the next few days.
    - Daily forecasts for planning or reporting.
    - Forecasts for a specific number of upcoming days.

    Args:
        n_days (int, optional):
            Number of future days to forecast.
            Defaults to 7.

    Returns:
        list[dict]:
            A list of daily forecast records. Each record contains:
            - date (Timestamp): Forecast date.
            - predicted_new_enterpriser (int): Predicted number of new enterprisers.
            - day_of_week (str): Name of the forecast day.

    Example:
        >>> forecast_next_n_days(3)
        [
            {
                "date": Timestamp("2026-06-20"),
                "predicted_new_enterpriser": 225,
                "day_of_week": "Saturday"
            },
            {
                "date": Timestamp("2026-06-21"),
                "predicted_new_enterpriser": 219,
                "day_of_week": "Sunday"
            },
            {
                "date": Timestamp("2026-06-22"),
                "predicted_new_enterpriser": 231,
                "day_of_week": "Monday"
            }
        ]
    """
    df = pd.read_csv(FEATURE_STORES, parse_dates=["date"], date_format="%Y-%m-%d")
    last_date = df["date"].max()
    forecasts = []
    model = joblib.load(MODEL_PATH)

    df_ext = df.copy()

    for i in range(n_days):
        future_date = last_date + pd.Timedelta(days=i + 1)
        new_row = pd.DataFrame([{
            'date': future_date,
            'new_enterpriser_count': np.nan,
            'new_bee_count': df_ext['new_bee_count'].median(),
            'is_promo_period': 0,
            'day_of_week': future_date.day_name(),
            'transaction_volume_online': df_ext['transaction_volume_online'].median(),
            'transaction_volume_offline': df_ext['transaction_volume_offline'].median(),
            'sales_ep_thousand_idr': df_ext['sales_ep_thousand_idr'].median(),
            'top_product_id': df_ext['top_product_id'].iloc[-1],
        }])
        df_ext = pd.concat([df_ext, new_row], ignore_index=True)
        df_feat = build_features(df_ext, drop_na=False)
        last_feat = df_feat[model.feature_name_ ].iloc[[-1]]
        pred = float(max(0, model.predict(last_feat)[0]))
        df_ext.loc[df_ext.index[-1], "new_enterpriser_count"] = pred

        forecasts.append({
            "date" : future_date,
            "predicted_new_enterpriser" : round(pred),
            "day_of_week" : future_date.day_name()
        })

    return forecasts


In [4]:
SYSTEM_PROMPT = """
You are HIVE (HDI Intelligence and Value Engine) — an internal AI business intelligence assistant for HDI executives, directors, and managers.

---

## Core Identity

You are NOT a general-purpose chatbot. You exist solely to transform business questions into data-driven insights using HDI's operational database.

**Always respond in the same language as the user's question.**

---

## Data Source

All answers must come from the HDI Daily Update database — your single source of truth.

Available metrics include (but are not limited to): new enterpriser registrations, transaction volume, EP sales, product performance, product ranking, and daily operational metrics.

**Data is available up to 2026-06-19. Never claim or imply data beyond this date.**

---

## Tools

### SQL Tool
Use for any query requiring historical business data. Never guess values the database can provide.

### Forecasting Tool
Use **only** when the user explicitly requests a prediction, forecast, or future estimate — and **only** for `new_enterpriser_count`.

For all other metrics, forecasting is not supported. Say so clearly.

---

## Response Format

Structure every response using the following sections:

- **Source Data** — Present the relevant data returned by the query in a concise table or list. Include only the information needed to support your analysis.
- **Executive Summary** — Provide a direct answer to the user's question.
- **Key Findings** — Summarize the main observations supported by the available data.
- **Actionable Insights** — Offer practical recommendations that are grounded in the evidence.

Keep the response professional, concise, and focused on business value. Avoid technical jargon unless the user explicitly requests it.

Do not fabricate information or make assumptions beyond the available data. If the query result is empty or insufficient, clearly state this instead of speculating.

Respond in the same language as the user's question.

---

## Analytical Standards

Always: use evidence, cite trends, apply percentages and growth rates, state assumptions  
Never: fabricate data, invent trends, assume causes without evidence

If data is insufficient, say so explicitly and ask a focused clarifying question.

---

Your mission: help HDI leadership make smarter decisions through accurate analysis and clear, evidence-based recommendations.
"""

In [19]:
MODEL_PRICING = {
    "gpt-4o-mini-2024-07-18": {
        "input": 0.15 / 1_000_000,
        "output": 0.60 / 1_000_000,
    },
    "gpt-4o-mini": {
        "input": 0.15 / 1_000_000,
        "output": 0.60 / 1_000_000,
    },
}

def get_agent(llm, tools):
    return create_agent(
        model=llm,
        system_prompt=SYSTEM_PROMPT,
        checkpointer=InMemorySaver(),
        tools=tools,
    )

def pretty_print_stream(events):
    for event in events:
        if event["type"] == "tool_call":
            print(f"\n[AI] Calling tool: {event['tool']}")

            args = event.get("args", {})
            if "query" in args:
                print(args["query"])
            else:
                print(pformat(args))

        elif event["type"] == "tool_result":
            print(f"\n[TOOL] {event['tool']}")
            print(event["content"])

        elif event["type"] == "final":
            print("\n========== ANSWER ==========")
            print(event["content"])
            print("============================\n")


def chatbot(query: str, config: dict, agent, stream: bool = False):
    start = time.perf_counter()

    usage = {
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
        "model": None,
    }

    def update_usage(message):
        usage_metadata = getattr(message, "usage_metadata", None)
        if usage_metadata:
            usage["prompt_tokens"] += usage_metadata.get("input_tokens", 0)
            usage["completion_tokens"] += usage_metadata.get("output_tokens", 0)
            usage["total_tokens"] += usage_metadata.get("total_tokens", 0)

        response_metadata = getattr(message, "response_metadata", None)
        if response_metadata:
            usage["model"] = response_metadata.get(
                "model_name",
                usage["model"],
            )

    def log_observability():
        elapsed = time.perf_counter() - start

        cost = 0.0
        model = usage["model"]

        if model in MODEL_PRICING:
            pricing = MODEL_PRICING[model]
            cost = (
                usage["prompt_tokens"] * pricing["input"]
                + usage["completion_tokens"] * pricing["output"]
            )

        logger.info(
            "\n========== OBSERVABILITY ==========\n"
            "Latency        : %.2f s\n"
            "Model          : %s\n"
            "Prompt Tokens  : %d\n"
            "Completion Tok.: %d\n"
            "Total Tokens   : %d\n"
            "Cost (USD)     : $%.6f\n"
            "===================================",
            elapsed,
            model,
            usage["prompt_tokens"],
            usage["completion_tokens"],
            usage["total_tokens"],
            cost,
        )

    if stream:

        def stream_events():
            for step in agent.stream(
                {
                    "messages": [HumanMessage(query)],
                },
                config=config,
            ):
                node_name = next(iter(step))
                node = step[node_name]

                for message in node.get("messages", []):

                    if message.type == "ai":
                        update_usage(message)

                        if message.tool_calls:
                            for tool in message.tool_calls:
                                yield {
                                    "type": "tool_call",
                                    "node": node_name,
                                    "tool": tool["name"],
                                    "args": tool["args"],
                                }

                        elif message.content:
                            yield {
                                "type": "final",
                                "node": node_name,
                                "content": message.content,
                            }

                    elif message.type == "tool":
                        yield {
                            "type": "tool_result",
                            "node": node_name,
                            "tool": message.name,
                            "content": message.content,
                        }

            log_observability()

        return stream_events()

    else:

        result = agent.invoke(
            {
                "messages": [HumanMessage(query)],
            },
            config=config,
        )

        for message in result["messages"]:
            if message.type == "ai":
                update_usage(message)

        log_observability()

        return result

In [ ]:
def get_engine_from_csv(csv_path, table_name="hdi_daily_ops"):
    df = pd.read_csv(csv_path, parse_dates=["date"])

    engine = sqlalchemy.create_engine(
        "sqlite://",
        connect_args={"check_same_thread": False},
        poolclass=sqlalchemy.pool.StaticPool,
    )

    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False,
    )

    return engine

def get_db_util(db_engine):
    return SQLDatabase(db_engine)

def ask(query: str, config: dict):
    llm = ChatOpenAI(model="gpt-4o-mini")
    engine = get_engine_from_csv(DATA_DIR / "raw" / "hdi_daily_ops.csv")
    db = get_db_util(engine)
    agent = get_agent(
        llm=llm,
        tools=get_sql_tools(db, llm) + [forecast_next_n_days],
    )
    events = chatbot(
        query=query,
        config=config,
        stream=True,
        agent=agent,
    )

    pretty_print_stream(events)

config = {
    "configurable": {
        "thread_id": str(uuid7())
    }
}

print("="*50)

ask(
    "Berapa total Enterpriser baru minggu ini menurut tanggal terakhir data di database?",
    config,
)

print("="*50)

ask(
    "Tunjukkan tren EP penjualan 30 hari terakhir",
    config,
)

print("="*50)

ask(
    "Hari apa dalam seminggu yang paling banyak registrasi Enterpriser?",
    config
)

print("="*50)

ask(
    "Prediksi jumlah Enterpriser baru 7 hari ke depan",
    config
)

print("="*50)


[AI] Calling tool: sql_db_query
SELECT SUM(new_enterpriser_count) AS total_new_enterprisers FROM daily_update WHERE date >= '2026-06-13' AND date <= '2026-06-19';

[TOOL] sql_db_query
Error: (sqlite3.OperationalError) no such table: daily_update
[SQL: SELECT SUM(new_enterpriser_count) AS total_new_enterprisers FROM daily_update WHERE date >= '2026-06-13' AND date <= '2026-06-19';]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[AI] Calling tool: sql_db_list_tables
{}

[TOOL] sql_db_list_tables
hdi_daily_ops

[AI] Calling tool: sql_db_query
SELECT SUM(new_enterpriser_count) AS total_new_enterprisers FROM hdi_daily_ops WHERE date >= '2026-06-13' AND date <= '2026-06-19';

[TOOL] sql_db_query
[(1655,)]

========== ANSWER ==========
- **Source Data**  
  | Total New Enterprisers |
  |------------------------|
  | 1655                   |

- **Executive Summary**  
  Total enterpriser baru yang terdaftar minggu ini (antara 13 Juni hingga 19 Juni 2026) adalah 1.655.

- **Key 